# Aprendizaje No Supervisado

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# Cargar variables de entorno
load_dotenv()
DB_USER = os.getenv('DB_USER')
DB_PASS = os.getenv('DB_PASS')
DB_HOST = '127.0.0.1'
DB_PORT = os.getenv('DB_PORT', '3306')
DB_NAME = os.getenv('DB_NAME')

# Crear el engine de conexión
DATABASE_URL = f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)

print(f"🔌 Conexión establecida con GeoLúmica DB: {DB_NAME}")

# --- CARGA COMPLETA DE TABLAS ---

# 1. Dimensiones
df_geo = pd.read_sql("SELECT * FROM dim_geografia", engine)
df_prov = pd.read_sql("SELECT * FROM dim_provincia", engine)

# 2. Tablas de Hechos (¡Ahora sí están todas!)
df_viirs = pd.read_sql("SELECT * FROM fact_viirs", engine)
df_demografia = pd.read_sql("SELECT * FROM fact_demografia", engine)
df_pib = pd.read_sql("SELECT * FROM fact_pib", engine)
df_migraciones = pd.read_sql("SELECT * FROM fact_migraciones", engine)
df_conectividad = pd.read_sql("SELECT * FROM fact_conectividad", engine)
df_consumo = pd.read_sql("SELECT * FROM fact_consumo_viviendas", engine)
df_osm = pd.read_sql("SELECT * FROM fact_osm", engine)
df_empresas = pd.read_sql("SELECT * FROM fact_empresas_transporte", engine) # ¡Añadido!

print(f"{len(df_geo)} municipios cargados en memoria.")
print(f" Dimensiones listas: Luz, Demografía, PIB, Migración, Conectividad, Consumo, OSM y Empresas.")

In [ ]:
print("\n" + "="*75)
print("📊 VERIFICACIÓN DE COBERTURA MULTI-DIMENSIONAL (DESDE MySQL)")
print("="*75)

# 1. Sacamos la lista oficial de los 8131 municipios del maestro
maestro_ids = set(df_geo['muni_key'].dropna().astype(str).str.zfill(5).unique())
total_master = len(maestro_ids)
print(f"Total de municipios en el maestro (dim_geografia): {total_master}\n")

# 2. Diccionario con todas las tablas de hechos a revisar
tablas_hechos = {
    "Demografía": df_demografia,
    "Luces (VIIRS)": df_viirs,
    "Renta/PIB": df_pib,
    "Migraciones": df_migraciones,
    "Conectividad": df_conectividad,
    "Transporte (OSM)": df_osm,
    "Empresas Logística": df_empresas,
    "Consumo Viviendas": df_consumo
}

# 3. Bucle de verificación
for nombre_tabla, df_tabla in tablas_hechos.items():
    # Extraemos los IDs únicos de esta tabla
    if 'muni_id_join' in df_tabla.columns:
        ids_tabla = set(df_tabla['muni_id_join'].dropna().astype(str).str.zfill(5).unique())
    else:
        print(f"▶ {nombre_tabla:<18} | ❌ ERROR: No tiene columna 'muni_id_join'")
        print("-" * 75)
        continue
        
    # Calculamos la intersección y la cobertura
    ids_encontrados = maestro_ids.intersection(ids_tabla)
    faltantes = maestro_ids - ids_tabla
    cobertura = (len(ids_encontrados) / total_master) * 100
    
    # Imprimimos el resultado formateado
    print(f"▶ {nombre_tabla:<18} | {len(ids_encontrados):>5} municipios vinculados | {cobertura:>6.2f}% cobertura")
    
    if cobertura == 100.0:
        print("  ✅ ¡PERFECTO! 100% de cobertura.")
    else:
        print(f"  ⚠️ Faltan {len(faltantes)} municipios.")
        # Opcional: mostrar un par de ejemplos de los que faltan cruzando con df_geo
        ejemplos_ids = list(faltantes)[:3]
        ejemplos_nombres = df_geo[df_geo['muni_key'].isin(ejemplos_ids)]['muni_display'].tolist()
        if ejemplos_nombres:
            print(f"  🔍 Ejemplos perdidos: {', '.join(ejemplos_nombres)}...")
            
    print("-" * 75)

## Feature Engineering y PCA

A partir de los datos del INE, calculamos:
* **`pob_absoluta_actual`**: Población total en el último año censado (escala del municipio).
* **`delta_pob_pct`**: Tasa de variación porcentual histórica. Valores negativos fuertes indican despoblación profunda.
* **`ratio_masculinidad`**: `(Hombres / Mujeres)`. En sociología rural, un valor inusualmente alto señala un núcleo en declive por la emigración de mujeres jóvenes.

In [ ]:
print("⚙️ [1/5] Creando Esqueleto y calculando Deltas Demográficos...")

# 1. Esqueleto Base
df_master = df_geo[['muni_key', 'muni_display', 'prov_id_join']].copy()
df_master = df_master.rename(columns={'muni_key': 'muni_id_join'})

# 2. Demografía
max_yr_demo = df_demografia['year'].max()
min_yr_demo = df_demografia['year'].min()

demo_actual = df_demografia[df_demografia['year'] == max_yr_demo][['muni_id_join', 'Total', 'Hombres', 'Mujeres']].rename(columns={'Total': 'pob_absoluta_actual'})
# Sumamos 0.1 a mujeres para evitar división por cero en pueblos abandonados
demo_actual['ratio_masculinidad'] = demo_actual['Hombres'] / (demo_actual['Mujeres'] + 0.1) 
demo_hist = df_demografia[df_demografia['year'] == min_yr_demo][['muni_id_join', 'Total']].rename(columns={'Total': 'pob_historica'})

demo_trend = pd.merge(demo_actual, demo_hist, on='muni_id_join', how='left')
demo_trend['delta_pob_pct'] = ((demo_trend['pob_absoluta_actual'] - demo_trend['pob_historica']) / (demo_trend['pob_historica'] + 1)) * 100

df_master = pd.merge(df_master, demo_trend[['muni_id_join', 'pob_absoluta_actual', 'delta_pob_pct', 'ratio_masculinidad']], on='muni_id_join', how='left')

* **`luz_absoluta_actual`**: Radiancia media del último año. Define el tamaño físico/lumínico de la zona.
* **`delta_luz_pct`**: Crecimiento de la luz a lo largo de los años. Si es alto pero la población no crece, indica expansión industrial/logística.
* **`luz_volatilidad_std`**: Desviación estándar mensual. Suavizamos los picos para detectar **estacionalidad** (zonas turísticas que brillan en verano y se apagan en invierno).

In [ ]:
print("⚙️ [2/5] Procesando Satélites VIIRS (Consumo y Volatilidad)...")

if 'year' not in df_viirs.columns:
    df_viirs['year'] = df_viirs['date'].str[:4].astype(int)

# Agrupamos los meses para sacar la media anual
viirs_anual = df_viirs.groupby(['muni_id_join', 'year'])['avg_rad'].mean().reset_index()
max_yr_viirs, min_yr_viirs = viirs_anual['year'].max(), viirs_anual['year'].min()

viirs_actual = viirs_anual[viirs_anual['year'] == max_yr_viirs][['muni_id_join', 'avg_rad']].rename(columns={'avg_rad': 'luz_absoluta_actual'})
viirs_hist = viirs_anual[viirs_anual['year'] == min_yr_viirs][['muni_id_join', 'avg_rad']].rename(columns={'avg_rad': 'luz_historica'})

viirs_trend = pd.merge(viirs_actual, viirs_hist, on='muni_id_join', how='left')
viirs_trend['delta_luz_pct'] = ((viirs_trend['luz_absoluta_actual'] - viirs_trend['luz_historica']) / (viirs_trend['luz_historica'] + 0.01)) * 100

# Volatilidad turística/estacional
viirs_std = df_viirs.groupby('muni_id_join')['avg_rad'].std().reset_index().rename(columns={'avg_rad': 'luz_volatilidad_std'})

df_master = pd.merge(df_master, viirs_trend[['muni_id_join', 'luz_absoluta_actual', 'delta_luz_pct']], on='muni_id_join', how='left')
df_master = pd.merge(df_master, viirs_std, on='muni_id_join', how='left')

Integramos los datos de la Agencia Tributaria sobre la renta disponible local.
* **`pib_absoluto_actual`**: Renta total del último año.
* **`pib_media_historica`**: Promedio de riqueza a lo largo del histórico.
* **`pib_volatilidad_std`**: Nivel de fluctuación de la economía (resiliencia frente a crisis).
* **Imputación**: A los municipios protegidos por el *Secreto Estadístico*, se les imputa automáticamente la media económica de su provincia.

In [ ]:
print("⚙️ [3/5] Integrando Riqueza e Imputando Secreto Estadístico...")

col_anio_pib = 'anio' if 'anio' in df_pib.columns else 'year'
max_yr_pib = df_pib[col_anio_pib].max()

pib_actual = df_pib[df_pib[col_anio_pib] == max_yr_pib][['muni_id_join', 'pib']].rename(columns={'pib': 'pib_absoluto_actual'})
pib_stats = df_pib.groupby('muni_id_join')['pib'].agg(pib_media_historica='mean', pib_volatilidad_std='std').reset_index()

df_master = pd.merge(df_master, pib_actual, on='muni_id_join', how='left')
df_master = pd.merge(df_master, pib_stats, on='muni_id_join', how='left')

# Imputación Inteligente (Media de la provincia para los 8 nulos de secreto estadístico)
for col in ['pib_absoluto_actual', 'pib_media_historica', 'pib_volatilidad_std']:
    df_master[col] = df_master.groupby('prov_id_join')[col].transform(lambda x: x.fillna(x.median()))

Combinamos variables de distintas fuentes para capturar el atractivo y la conectividad del territorio. Para normalizar los datos y evitar que el tamaño del municipio domine el modelo, usamos tasas y variaciones:

* **`delta_empresas_transporte_pct`**: Crecimiento del tejido logístico. *Cálculo: `((Empresas_Actuales - Empresas_Historicas) / Empresas_Historicas) * 100`*. Picos masivos (+400%) en zonas no metropolitanas detectan nuevos grandes nodos logísticos periféricos (el "Efecto Amazon").
* **`stations_density_km2`**: Nivel de infraestructura ferroviaria metropolitana por kilómetro cuadrado. Diferencia los núcleos urbanos densos del resto.
* **`Indice_Conectividad`**: Nivel de motorización general (vehículos).
* **`Pct_Vehiculos_Muni_vs_Prov`**: Porcentaje del parque automotor de la provincia que se concentra en este municipio específico. Detecta las "capitales de facto" en términos de movilidad.
* **`tasa_migratoria_pct`**: Tasa de presión migratoria. *Cálculo: `(Migrantes Totales / Población Total) * 100`*. Transforma el volumen bruto en una "presión de atracción" comparable, revelando si un pueblo pequeño atrae a mucha gente en proporción a su tamaño.

In [ ]:
print("⚙️ [4/5] Fusionando Empresas, Transporte (OSM) y Magnetismo Migratorio...")

# 1. Empresas de Logística (Cálculo del "Efecto Amazon")
year_cols = [c for c in df_empresas.columns if c.isdigit()]
if year_cols:
    max_yr_emp, min_yr_emp = max(year_cols), min(year_cols)
    df_empresas['emp_act'] = pd.to_numeric(df_empresas[max_yr_emp], errors='coerce')
    df_empresas['emp_hist'] = pd.to_numeric(df_empresas[min_yr_emp], errors='coerce')
    
    # Variación porcentual histórica
    df_empresas['delta_empresas_transporte_pct'] = ((df_empresas['emp_act'] - df_empresas['emp_hist']) / (df_empresas['emp_hist'] + 1)) * 100
    df_master = pd.merge(df_master, df_empresas[['muni_id_join', 'emp_act', 'delta_empresas_transporte_pct']].rename(columns={'emp_act': 'empresas_transporte_actual'}), on='muni_id_join', how='left')

# 2. Conectividad y OSM (Incluyendo Pct_Vehiculos)
col_anio_con = 'Anio' if 'Anio' in df_conectividad.columns else 'year'
max_con = df_conectividad[col_anio_con].max()
con_actual = df_conectividad[df_conectividad[col_anio_con] == max_con][['muni_id_join', 'Indice_Conectividad', 'Pct_Vehiculos_Muni_vs_Prov']]
df_master = pd.merge(df_master, con_actual, on='muni_id_join', how='left')

df_master = pd.merge(df_master, df_osm[['muni_id_join', 'stations_density_km2', 'mean_distance_km_to_station']], on='muni_id_join', how='left')

# 3. Migración (Cálculo de la Tasa Migratoria)
col_anio_mig = 'anio' if 'anio' in df_migraciones.columns else 'year'
max_mig = df_migraciones[col_anio_mig].max()
mig_actual = df_migraciones[(df_migraciones[col_anio_mig] == max_mig) & (df_migraciones['nacionalidad'].str.lower() == 'total')]

df_master = pd.merge(df_master, mig_actual[['muni_id_join', 'cantidad (personas)']].rename(columns={'cantidad (personas)': 'mig_act'}), on='muni_id_join', how='left')

# Tasa = (Migrantes / Población Total) * 100
df_master['tasa_migratoria_pct'] = (df_master['mig_act'] / (df_master['pob_absoluta_actual'] + 1)) * 100
df_master = df_master.drop(columns=['mig_act']) # Borramos el dato bruto para no confundir al modelo

Tratamiento matemático final de la matriz antes de inyectarla en los algoritmos:
* **Penalización de Aislamiento**: A los municipios sin datos de distancia ferroviaria en OSM se les imputa un valor extremo (`100.0` km) para que el modelo los segregue correctamente como "Aislados".
* El resto de valores residuales nulos (pueblos minúsculos sin datos de empresas) se rellenan con `0`.

In [ ]:
print("⚙️ [5/5] Limpieza de Nulos Residuales (Penalizaciones Matemáticas)...")

# Penalizamos los pueblos de montaña / aislados sin estación de OSM con 100km
df_master['mean_distance_km_to_station'] = df_master['mean_distance_km_to_station'].fillna(100.0)

# El resto de variables no mapeadas (pueblos sin registros de empresas) a cero
df_master = df_master.fillna(0)

print("\n" + "="*80)
print(f"✅ ¡MATRIZ MAESTRA LISTA PARA PCA! Dimensiones: {df_master.shape[0]} filas x {df_master.shape[1]} columnas.")
print("="*80)
display(df_master.head()) # Usamos display() que se ve mucho mejor en Jupyter que print()

## Modelos